In [13]:
from getpass import getpass
from openai import OpenAI

# API 키를 코드에 적지 않고 실행할 때 입력합니다.
api_key = getpass("발급받은 OpenAI API 키를 입력하세요: ").strip()

# 첫 연결 확인에서는 자동 재시도를 사용하지 않습니다.
client = OpenAI(api_key=api_key, max_retries=0)

# 짧은 테스트 요청을 한 번 보냅니다.
response = client.responses.create(
    model="gpt-6-luna",
    input="'API 연결에 성공했습니다.'라고 한 문장만 답해주세요.",
    reasoning={"effort": "none"},
    max_output_tokens=128,
)

# API 키가 아니라 모델의 응답만 출력합니다.
print(response.output_text)

API 연결에 성공했습니다.


In [14]:
OPENAI_API_KEY = api_key

In [15]:
from openai import OpenAI
from dotenv import load_dotenv


# 1. 시스템 프롬프트
system_prompt = """
당신은 명언을 추천하는 AI입니다.

제공된 참고 문서만 사용하여 답변하세요.
질문에 적합한 명언 하나를 추천하세요.
명언의 문서 ID와 추천 이유를 함께 작성하세요.
"""

# 2. 검색 결과라고 가정한 가상의 문서
documents = [
    {
        "id": "Q01",
        "text": "Small steps can lead to progress."
    },
    {
        "id": "Q02",
        "text": "Mistakes are opportunities to learn and try again."
    },
    {
        "id": "Q03",
        "text": "Kindness helps build strong relationships."
    }
]

# 3. 참고 문맥 구성
context = "\n".join(
    f"{doc['id']}: {doc['text']}"
    for doc in documents
)

# 4. 사용자 질문
user_question = """
실패했을 때 다시 도전하는 데
도움이 되는 명언을 추천해 줘.
"""

# 5. 사용자 프롬프트 구성
user_prompt = f"""
[참고 문서]
{context}

[사용자 질문]
{user_question}
"""

# 6. API에 전달할 메시지 구성
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

# 7. 실제 전달할 메시지 내용 확인
print("=== API 입력 메시지 ===")

for message in messages:
    print(f"\n[{message['role']}]")
    print(message["content"].strip())

load_dotenv()

client = OpenAI(api_key=api_key, max_retries=0)

response = client.responses.create(
    model="gpt-5.6-luna",
    input=messages
)

print("\n=== LLM 응답 ===")
print(response.output_text)

=== API 입력 메시지 ===

[system]
당신은 명언을 추천하는 AI입니다.

제공된 참고 문서만 사용하여 답변하세요.
질문에 적합한 명언 하나를 추천하세요.
명언의 문서 ID와 추천 이유를 함께 작성하세요.

[user]
[참고 문서]
Q01: Small steps can lead to progress.
Q02: Mistakes are opportunities to learn and try again.
Q03: Kindness helps build strong relationships.

[사용자 질문]

실패했을 때 다시 도전하는 데
도움이 되는 명언을 추천해 줘.

=== LLM 응답 ===
명언: “Mistakes are opportunities to learn and try again.”  
문서 ID: Q02  
추천 이유: 실수를 실패로만 보지 않고, 배우고 다시 도전할 기회로 바라보게 해주는 명언입니다.


In [16]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=api_key, max_retries=0)

# 분석할 고객 리뷰
user_prompt = """
주문번호 A-1042로 주문한 무선 마우스가
예정일보다 3일 늦게 도착했습니다.
제품은 정상적으로 작동하지만,
고객센터에 문의해도 답변이 없어 불편했습니다.
"""

# 분석 목적별 프롬프트
system_prompts = {
    "정보 추출": """
[Role]
당신은 고객 문의 분석 담당자입니다.

[Instruction]
주문번호, 상품명, 문제 상황을 추출하세요.

[Constraint]
원문에 없는 정보를 추가하지 마세요.
확인할 수 없는 정보는 '정보 없음'으로 표시하세요.

[Output Format]
주문번호: ...
상품명: ...
문제 상황: ...
""",

    "텍스트 분류": """
[Role]
당신은 고객 문의 분류 담당자입니다.

[Instruction]
문의의 주요 내용을 분류하세요.

[Constraint]
배송, 환불, 상품 품질, 기타 중 하나만 선택하세요.

[Output Format]
분류: ...
""",

    "텍스트 요약": """
[Role]
당신은 고객 리뷰 요약 담당자입니다.

[Instruction]
고객 리뷰의 핵심 내용을 요약하세요.

[Constraint]
원문에 없는 정보를 추가하지 마세요.
최대 3문장으로 작성하세요.

[Output Format]
요약: ...
"""
}

# 분석 작업별 API 호출
for task, system_prompt in system_prompts.items():

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    print(f"=== {task} ===")
    print(response.output_text)
    print()

=== 정보 추출 ===
주문번호: A-1042  
상품명: 무선 마우스  
문제 상황: 예정일보다 3일 늦게 도착했으며, 고객센터에 문의했지만 답변이 없어 불편함. 제품은 정상 작동함.

=== 텍스트 분류 ===
분류: 배송

=== 텍스트 요약 ===
요약: 주문번호 A-1042의 무선 마우스가 예정일보다 3일 늦게 도착했습니다. 제품은 정상 작동했지만, 고객센터의 답변이 없어 불편을 겪었습니다.



In [20]:
OPENAI_API_KEY = api_key

In [ ]:
import os
import json

from dotenv import load_dotenv
from openai import OpenAI, OpenAIError

# 1. API Key 및 클라이언트 준비
load_dotenv(r'C:\text_processing\LLM\.env')

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY를 확인하세요.")

client = OpenAI(
    timeout=30.0,
    max_retries=0
)

# 분석 대상
quote = "Small steps can lead to big changes."

allowed_categories = {"성장", "관계", "기타"}

# 2. 구조화된 출력 스키마
category_field = {
    "type": "string",
    "enum": ["성장", "관계", "기타"]
}

# 주제 분류용
category_schema = {
    "type": "object",
    "properties": {
        "category": category_field
    },
    "required": ["category"],
    "additionalProperties": False
}

# 주제 분류 + 요약용
analysis_schema = {
    "type": "object",
    "properties": {
        "category": category_field,
        "summary": {"type": "string"}
    },
    "required": ["category", "summary"],
    "additionalProperties": False
}

# 3. 공통 API 호출 함수
def call_llm(instructions, input_text, schema=None):

    # 기본 출력은 일반 텍스트
    text_option = {
        "format": {"type": "text"}
    }

    # 스키마가 있으면 구조화된 출력 사용
    if schema is not None:
        text_option = {
            "format": {
                "type": "json_schema",
                "name": "quote_result",
                "strict": True,
                "schema": schema
            }
        }

    response = client.responses.create(
        model="gpt-6-luna",
        reasoning={"effort": "none"},
        instructions=instructions,
        input=input_text,
        temperature=0,
        max_output_tokens=200,
        store=False,
        text=text_option
    )

    # 응답 상태 확인
    if response.status != "completed":
        raise ValueError(
            f"응답 미완료: {response.status}"
        )

    if not response.output_text:
        raise ValueError("출력 없음 또는 요청 거부")

    return response.output_text.strip()


# 4. 공통 주제 분류 함수
def classify_quote(text):

    output_text = call_llm(
        instructions=(
            "명언을 다음 기준으로 분류하세요.\n"
            "성장: 노력, 학습, 변화, 자기계발\n"
            "관계: 친절, 우정, 인간관계\n"
            "기타: 위 두 주제에 해당하지 않는 경우"
        ),
        input_text=text,
        schema=category_schema
    )

    result = json.loads(output_text)
    category = result["category"]

    if category not in allowed_categories:
        raise ValueError("허용되지 않은 분류 결과")

    return category


# 5. 단일 호출 방식
def single_workflow(text):

    output_text = call_llm(
        instructions=(
            "명언의 주제를 성장, 관계, 기타 중 "
            "하나로 분류하세요.\n"
            "성장: 노력, 학습, 변화, 자기계발\n"
            "관계: 친절, 우정, 인간관계\n"
            "기타: 위 두 주제에 해당하지 않는 경우\n"
            "핵심 의미를 한국어 한 문장으로 요약하세요."
        ),
        input_text=text,
        schema=analysis_schema
    )


## category와 summary 두개 값 가진다. 

    result = json.loads(output_text)

    if (
        result["category"] not in allowed_categories ## 허용이 되는가
        or not result["summary"].strip()  ## summary가 비어 있지 않냐
    ):
        raise ValueError("분석 결과 검증 실패")

    return result


# 6. 순차 처리 방식
def sequential_workflow(text):

    # 1단계: 주제 분류
    category = classify_quote(text)   # category를 하나 뽑는다. 
    print("1단계 - 주제 분류:", category)

    # 2단계: 분류 결과를 참고하여 요약
    summary = call_llm(                             ### summary도 하나 뽑는다. 
        instructions=(
            "제공된 명언의 원문을 근거로 "
            "핵심 의미를 한국어 한 문장으로 요약하세요. "
            "앞서 분류된 주제를 참고하세요."
        ),
        input_text=(
            f"명언: {text}\n"
            f"분류 결과: {category}"
        )
    )   ## 이거는 system prompt와 user prompt를 뽑는거


    print("2단계 - 주제 기반 요약:", summary)

    return {
        "category": category,
        "summary": summary
    }


# 7. 조건부 분기 방식
def conditional_workflow(text):

    # 1단계: 주제 분류
    category = classify_quote(text)
    print("1단계 - 주제 분류:", category)

    # 2단계: 분류 결과에 따라 분석 작업 선택
    if category == "성장":
        task = "실천할 수 있는 행동 한 가지를 제안하세요."

    elif category == "관계":
        task = "인간관계에 관한 교훈을 설명하세요."

    elif category == "기타":
        task = "명언의 일반적인 핵심 의미를 요약하세요."

    else:
        raise ValueError("분기 조건에 없는 분류 결과")

    print("2단계 - 선택된 작업:", task)

    # 3단계: 선택된 분석 작업 실행
    message = call_llm(
        instructions=(
            f"{task} "
            "명언의 원문을 근거로 한국어 한 문장으로 작성하세요."
        ),
        input_text=text
    )

    print("3단계 - 후속 분석:", message)

    return {
        "category": category,
        "task": task,
        "message": message
    }


# 8. 세 가지 워크플로우 실행 및 결과 확인
workflows = [
    ("단일 호출", single_workflow),
    ("순차 처리", sequential_workflow),
    ("조건부 분기", conditional_workflow)
]

for name, workflow in workflows:

    print(f"\n[{name}]")

    try:
        result = workflow(quote)
        print("최종 결과:", result)

    except (OpenAIError, ValueError, KeyError) as e:
        print("워크플로우 처리 실패:", e)


[단일 호출]
최종 결과: {'category': '성장', 'summary': '작은 노력과 변화가 쌓이면 큰 변화를 이룰 수 있다.'}

[순차 처리]
1단계 - 주제 분류: 성장
2단계 - 주제 기반 요약: 작은 실천을 꾸준히 이어가면 큰 성장과 변화를 이룰 수 있다.
최종 결과: {'category': '성장', 'summary': '작은 실천을 꾸준히 이어가면 큰 성장과 변화를 이룰 수 있다.'}

[조건부 분기]
1단계 - 주제 분류: 성장
2단계 - 선택된 작업: 실천할 수 있는 행동 한 가지를 제안하세요.
3단계 - 후속 분석: “Small steps can lead to big changes”라는 말처럼, 오늘 목표를 이루기 위해 지금 할 수 있는 작은 행동 하나를 시작해 보세요.
최종 결과: {'category': '성장', 'task': '실천할 수 있는 행동 한 가지를 제안하세요.', 'message': '“Small steps can lead to big changes”라는 말처럼, 오늘 목표를 이루기 위해 지금 할 수 있는 작은 행동 하나를 시작해 보세요.'}


single workflow 기준으로 봤을때 LLM 이 보는것

입력
   output_text = call_llm(
        instructions=(
            "명언의 주제를 성장, 관계, 기타 중 "
            "하나로 분류하세요.\n"
            "성장: 노력, 학습, 변화, 자기계발\n"
            "관계: 친절, 우정, 인간관계\n"
            "기타: 위 두 주제에 해당하지 않는 경우\n"
            "핵심 의미를 한국어 한 문장으로 요약하세요."
        ),
        input_text=text,
        schema=analysis_schema
    )
결과

명언에 대한 category 와 명언에 대한 summary


squential workflow

 # 1단계: 주제 분류
    category = classify_quote(text)   # category를 하나 뽑는다. 
    print("1단계 - 주제 분류:", category)

    # 2단계: 분류 결과를 참고하여 요약
    summary = call_llm(                             ### summary도 하나 뽑는다. 
        instructions=(
            "제공된 명언의 원문을 근거로 "
            "핵심 의미를 한국어 한 문장으로 요약하세요. "
            "앞서 분류된 주제를 참고하세요."
        ),
        input_text=(
            f"명언: {text}\n"
            f"분류 결과: {category}"
        )
    )   ## 이거는 system prompt와 user prompt를 뽑는거


    print("2단계 - 주제 기반 요약:", summary)

    return {
        "category": category,
        "summary": summary
    }



sequential workflow
1번 LLM
입력: 분류해주세요
     명언
출력 : 범주중 1개

2번 LLM
입력:
"제공된 명언의 원문을 근거로 "
            "핵심 의미를 한국어 한 문장으로 요약하세요. "
            "앞서 분류된 주제를 참고하세요."
    f"명언: {text}\n"
            f"분류 결과: {category}"
출력: 요약




conditional workflow


1번 LLM
입력 : 분류해주세요
       명언
출력: 범주 중 1개

2번 LLM
- 1번 LLM == 성장
-- 2번 LLM의 입력
"실천할 수 있는 행동 한가지를 제안하세요"

- 1번 LLM ==  관계
-- 2번 LLM LLM의 입력
"인간관계에 관한 교훈을 설명하세요."


- 1번 LLM ==  기타
-- 2번 LLM LLM의 입력
 "명언의 일반적인 핵심 의미를 요약하세요.